# Codi gen DATA_5 Psychiatric_diagnosis

## Psychiatric diseases with code ICD-10:
| Catalan / Spanish | English Translation | ICD-10 Range | Prevalence (Applied) |
| :--- | :--- | :--- | :--- |
| **Trastorns de l'Ansietat** | **Anxiety Disorders** | F40–F48 | 31.38% |
| **Trastorns de l'Ànim / de l'Humor** | **Mood (Affective) Disorders** | F30–F39 | 18.83% |
| **Esquizofrènia i T. Psicòtics** | **Schizophrenia and Psychotic Disorders** | F20–F29 | 3.50% |
| **Trastorns per ús de substàncies** | **Substance Use Disorders** | F10–F19 | 10.89% |

In [8]:
import pandas as pd
import numpy as np
import random

In [13]:
path_pacients = r'C:\Users\LRAJAG\Desktop\Data_sintetica\csv\1_Sociodemographic\pacients_100k_sintetic.csv'
df_original = pd.read_csv(path_pacients)

In [22]:
# 1. Configuració de Rangs, Probabilitats i Noms
# Aquestes probabilitats determinen la "sort" de cada pacient de ser diagnosticat
config_diagnostics = [
    {'nom': 'Anxiety', 'rang': (40, 48), 'prob': 0.3138, 'desc': "Anxiety disorder"},
    {'nom': 'Mood', 'rang': (30, 39), 'prob': 0.1883, 'desc': "Mood disorder"},
    {'nom': 'Substance', 'rang': (10, 19), 'prob': 0.1359, 'desc': "Substance use disorder"},
    {'nom': 'Psychotic', 'rang': (20, 29), 'prob': 0.035, 'desc': "Psychotic disorder"}
]

# 2. Carregar la teva llista de 100k IDs
df_5 = df_original[['id']].copy()

# 3. Procés de DIAGNÒSTIC (Simulació activa)
np.random.seed(42)
random.seed(42)

def aplicar_diagnostics(row):
    codis = []
    noms = []
    
    # Intentem diagnosticar segons probabilitat normal
    for d in config_diagnostics:
        if np.random.rand() < d['prob']:
            num = random.randint(d['rang'][0], d['rang'][1])
            codis.append(f"F{num}")
            noms.append(d['desc'])
    
    # SI NO HA SORTIT CAP (Pacient "Sa"), n'assignem un de principal obligatòriament
    if not codis:
        # Triem una categoria a l'atzar basada en el pes de les probabilitats
        categories = config_diagnostics
        pesos = [d['prob'] for d in categories]
        # Normalitzem els pesos perquè sumin 1
        pesos_norm = [p / sum(pesos) for p in pesos]
        
        principal = np.random.choice(categories, p=pesos_norm)
        num = random.randint(principal['rang'][0], principal['rang'][1])
        codis.append(f"F{num}")
        noms.append(principal['desc'])
    
    return pd.Series([", ".join(codis), ", ".join(noms)])

# Apliquem el diagnòstic a cada fila
df_5[['codi_cie10', 'nom_diagnostic_complet']] = df_5.apply(aplicar_diagnostics, axis=1)

# 4. Guardar el resultat
path_sortida = r'C:\Users\LRAJAG\Desktop\Data_sintetica\csv\5_Diagnose\diagnostics_psiquiatrics.csv'
df_5.to_csv(path_sortida, index=False)

print("Diagnòstic completat per als 100.000 pacients.")
print(df_5.head(15))

Diagnòstic completat per als 100.000 pacients.
    id codi_cie10                 nom_diagnostic_complet
0    1        F41                       Anxiety disorder
1    2   F40, F34        Anxiety disorder, Mood disorder
2    3        F33                          Mood disorder
3    4   F43, F32        Anxiety disorder, Mood disorder
4    5        F41                       Anxiety disorder
5    6        F48                       Anxiety disorder
6    7        F41                       Anxiety disorder
7    8   F39, F16  Mood disorder, Substance use disorder
8    9        F30                          Mood disorder
9   10   F30, F21      Mood disorder, Psychotic disorder
10  11        F33                          Mood disorder
11  12        F33                          Mood disorder
12  13        F48                       Anxiety disorder
13  14   F40, F38        Anxiety disorder, Mood disorder
14  15        F43                       Anxiety disorder


## Afegir columna diagnòstic específic

In [24]:
# 1. El teu diccionari de traduccions reals (CIE-10)
diccionari_cie10 = {
    # Anxiety (F40-F48)
    'F40': 'Phobic anxiety disorders',
    'F41': 'Other anxiety disorders (generalized/panic)',
    'F42': 'Obsessive-compulsive disorder',
    'F43': 'Reaction to severe stress, and adjustment disorders',
    'F44': 'Dissociative (conversion) disorders',
    'F45': 'Somatoform disorders',
    'F46': 'Other anxiety disorders (unspecified)',
    'F47': 'Other anxiety disorders (unspecified)',
    'F48': 'Other neurotic disorders',
    
    # Mood (F30-F39)
    'F30': 'Manic episode',
    'F31': 'Bipolar affective disorder',
    'F32': 'Depressive episode',
    'F33': 'Recurrent depressive disorder',
    'F34': 'Persistent mood (affective) disorders (cyclothymia/dysthymia)',
    'F35': 'Other mood (affective) disorders (unspecified)', 
    'F36': 'Other mood (affective) disorders (unspecified)',
    'F37': 'Other mood (affective) disorders (unspecified)',
    'F38': 'Other mood (affective) disorders',
    'F39': 'Unspecified mood (affective) disorder',
    
    # Substance Use (F10-F19)
    'F10': 'Mental and behavioral disorders due to use of alcohol',
    'F11': 'Mental and behavioral disorders due to use of opioids',
    'F12': 'Mental and behavioral disorders due to use of cannabinoids',
    'F13': 'Mental and behavioral disorders due to use of sedatives or hypnotics',
    'F14': 'Mental and behavioral disorders due to use of cocaine',
    'F15': 'Mental and behavioral disorders due to use of other stimulants',
    'F16': 'Mental and behavioral disorders due to use of hallucinogens',
    'F17': 'Mental and behavioral disorders due to use of tobacco',
    'F18': 'Mental and behavioral disorders due to use of volatile solvents',
    'F19': 'Mental and behavioral disorders due to multiple drug use',
    
    # Schizophrenia and Psychotic Disorders (F20-F29)
    'F20': 'Schizophrenia',
    'F21': 'Schizotypal disorder',
    'F22': 'Persistent delusional disorders',
    'F23': 'Acute and transient psychotic disorders',
    'F24': 'Induced delusional disorder',
    'F25': 'Schizoaffective disorders',
    'F26': 'Other nonorganic psychotic disorders',
    'F27': 'Other nonorganic psychotic disorders',
    'F28': 'Other nonorganic psychotic disorders',
    'F29': 'Unspecified nonorganic psychosis'
}


path_fitxer = r'C:\Users\LRAJAG\Desktop\Data_sintetica\csv\5_Diagnose\diagnostics_psiquiatrics.csv'
df_5_2 = pd.read_csv(path_fitxer)

# 3. Funció per buscar el nom específic basant-se en el codi
def cercar_nom_especific(text_codis):
    if pd.isna(text_codis) or text_codis == "Sense diagnòstic":
        return "Sense diagnòstic específic"
    
    # Separem per comes per si hi ha comorbiditat (diversos codis)
    llista_codis = [c.strip() for c in str(text_codis).split(',')]
    
    # Mapegem cada codi al seu nom real del diccionari
    noms = [diccionari_cie10.get(c, "Codi no trobat") for c in llista_codis]
    
    return ", ".join(noms)

# 4. Afegim la 4a columna "diagnostic_especific"
# Mantenim intactes id_pacient, codi_cie10 i la columna de grup original
df_5_2['diagnostic_especific'] = df_5_2['codi_cie10'].apply(cercar_nom_especific)

# 5. Guardem el fitxer actualitzat
df_5_2.to_csv(path_fitxer, index=False)

print("Fitxer actualitzat amb 4 columnes:")
print(df_5_2.columns.tolist())
print("\nExemple de les dades:")
print(df_5_2[['id', 'codi_cie10', 'nom_diagnostic_complet', 'diagnostic_especific']].head(10))

Fitxer actualitzat amb 4 columnes:
['id', 'codi_cie10', 'nom_diagnostic_complet', 'diagnostic_especific']

Exemple de les dades:
   id codi_cie10                 nom_diagnostic_complet  \
0   1        F41                       Anxiety disorder   
1   2   F40, F34        Anxiety disorder, Mood disorder   
2   3        F33                          Mood disorder   
3   4   F43, F32        Anxiety disorder, Mood disorder   
4   5        F41                       Anxiety disorder   
5   6        F48                       Anxiety disorder   
6   7        F41                       Anxiety disorder   
7   8   F39, F16  Mood disorder, Substance use disorder   
8   9        F30                          Mood disorder   
9  10   F30, F21      Mood disorder, Psychotic disorder   

                                diagnostic_especific  
0        Other anxiety disorders (generalized/panic)  
1  Phobic anxiety disorders, Persistent mood (aff...  
2                      Recurrent depressive disorder  
3